In [0]:
%sql

SELECT 
description AS appt_desciption, COUNT(DISTINCT sch_appt_id) AS appt_count
FROM 4_prod.raw.mill_sch_appt AS sa
WHERE
description LIKE '%CARDIO%'
GROUP BY description
ORDER BY appt_count DESC

In [0]:
%sql

SELECT Clinic, COUNT(DISTINCT sch_appt_id) AS Count, Tag, COUNT(DISTINCT en.person_id), COUNT(en.person_id)
FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
INNER JOIN 4_prod.raw.mill_sch_appt AS sa
ON fc.clinic = sa.description
LEFT JOIN 4_prod.raw.mill_sch_event AS se
ON sa.sch_event_id = se.sch_event_id
LEFT JOIN 4_prod.raw.mill_encounter AS en
ON se.encntr_id = en.encntr_id
GROUP BY clinic, tag
ORDER BY tag, COUNT(sch_appt_id) DESC

In [0]:
%sql
SELECT * FROM 4_prod.raw.mill_encounter WHERE encntr_id = 0

In [0]:
%sql
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT *
    FROM a1
    UNION
    SELECT *
    FROM a2
)
SELECT Tag, COUNT(DISTINCT person_id) AS PatientCount
FROM u
GROUP BY Tag

In [0]:
%sql
CREATE TABLE 5_projects.dar064.rde_patient_demographics AS
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT DISTINCT person_id
    FROM a1
    UNION
    SELECT DISTINCT person_id
    FROM a2
)
SELECT pd.* EXCEPT (NHS_Number, MRN, Date_of_Birth)
FROM u
INNER JOIN 4_prod.rde.rde_patient_demographics AS pd
ON u.PERSON_ID = pd.PERSON_ID


In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT PERSON_ID) FROM 5_projects.dar064.rde_patient_demographics

In [0]:
from tqdm.notebook import tqdm

table_names = ["rde_apc_diagnosis", "rde_apc_opcs", "rde_ariapharmacy", "rde_blobdataset", "rde_cds_apc", "rde_cds_opa", "rde_emergency_findings", "rde_emergencyd", "rde_encounter", "rde_measurements", "rde_medadmin", "rde_op_diagnosis", "rde_opa_opcs", "rde_pathology", "rde_pharmacyorders", "rde_powerforms", "rde_radiology", "rde_raw_pathology"]

for table_name in tqdm(table_names):
    q = f"""CREATE OR REPLACE TABLE 5_projects.dar064.{table_name} AS
    SELECT t.*
    FROM 4_prod.rde.{table_name} AS t
    INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
    ON t.person_id = pd.person_id"""
    spark.sql(q)